# Station template matching: local working notebook for review

Imported from Rainier_notebook/templates-seismometers.ipynb; code is unchanged and saved outputs are removed. This is **not a verified final production workflow**. It mixes exploratory station lists, four-West examples, server paths, and an older hardcoded 30-unmatched station summary. Confirm the five-station production run and final 1000-km screening before using it to reproduce manuscript counts. Do not Run All without reviewing paths and network/download cells.


## 1. Load scientific libraries

Imports support waveform retrieval, filtering, correlation and plotting. Install these dependencies first; importing them does not download data.


In [ ]:
import obspy
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import matplotlib.pyplot as plt
import os
import numpy as np
from scipy.signal import butter, filtfilt
import numpy as np
import pandas as pd
import matplotlib.dates as mdates 
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import seaborn as sns
from obspy import UTCDateTime, Stream
from obspy import UTCDateTime
from datetime import datetime, timedelta
import numpy as np
from libcomcat.search import search
from libcomcat.dataframes import get_summary_data_frame
from obspy.clients.fdsn import Client
from obspy import UTCDateTime, Stream
from obspy.geodetics.base import gps2dist_azimuth  #
import glob

## 2. Choose template-preparation stations

This exploratory list contains 11 vertical channels. It is not evidence that all 11, or the four channels used in later examples, produced the final five-station manuscript catalog. Keep these roles separate.


In [ ]:
#  Creating templates with this stations
path = '/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/'


client = Client("IRIS")

# Your original station definitions
#list_stations = [
#    'UW-FMW-EHZ','UW-FMW-HHZ','UW-LON-BHZ','UW-LON-EHZ','UW-LON-HHZ',
#    'UW-RCM-EHZ','UW-RCM-HHZ','UW-RCS-EHZ','UW-RER-EHZ','UW-RER-HHZ',
#    'UW-RVC-EHZ','UW-WPW-EHZ','CC-ARAT-BHZ','CC-COPP-BHZ','CC-GOBB-BHZ',
#    'CC-KAUT-BHZ','CC-MILD-BHZ','CC-MIRR-BHZ','CC-OBSR-BHZ','CC-PANH-BHZ',
#    'CC-PARA-BHZ','CC-RUSH-BHZ','CC-SR41-EHZ','CC-TABR-BHZ','CC-TAVI-BHZ'
#]

list_stations = [
    'CC-KAUT-BHZ', 'CC-MILD-BHZ', 'CC-OBSR-BHZ', 'CC-PANH-BHZ',
    'CC-PARA-BHZ', 'CC-RUSH-BHZ', 'UW-FMW-HHZ', 'UW-LON-HHZ',
    'UW-RCM-HHZ', 'UW-RCS-EHZ', 'UW-RER-HHZ']

## 3. Retrieve catalog events and inspect waveform availability

Query ComCat for the specified August swarm interval and download an example event. Filtering and peak normalization here facilitate waveform comparison; they do not measure absolute amplitudes. This cell uses network access.


In [ ]:
# 1. Catalog search (ComCat)
events = search(
    starttime=datetime(2023, 8, 25, 0, 0),
    endtime=datetime(2023, 8, 31, 0, 0),
    latitude=46.879967,
    longitude=-121.726906,
    maxradius=35/111.32,  # 35 km in degrees
)
event_df = get_summary_data_frame(events)
print(f"Returned {len(events)} events")

# 2. Pick one event origin time
oritime = event_df["time"]          # pandas Series
timeexample = oritime.iloc[0]       # first event time (Timestamp)

# Convert to ObsPy UTCDateTime
start_time = UTCDateTime(timeexample)   # IMPORTANT conversion
end_time   = start_time + 1* 60       # 10 minutes after origin

# 3. Waveform download parameters
freqmin, freqmax, corners = 2, 10, 4

client = Client("IRIS")  # or "NCEDC", "USGS", etc.

st_big = Stream()
have, nohave = [], []

for sta_str in list_stations:   # e.g. "UW-ASR-HHZ"
    net, sta, cha = sta_str.split("-")
    try:
        st = client.get_waveforms(net, sta, "*", cha, start_time, end_time)
    except Exception as e:
        print(f"{sta_str}: waveform request failed: {e}")
        nohave.append(sta_str)
        continue

    if len(st) == 0:
        nohave.append(sta_str)
        continue

    # Merge gaps and detrend
    st.merge(fill_value="interpolate")
    st.detrend("linear")

    added = False
    for tr in st:
        # Bandpass 2–10 Hz, zero-phase
        tr.filter(
            "bandpass",
            freqmin=freqmin,
            freqmax=freqmax,
            corners=corners,
            zerophase=True,
        )

        # Peak-normalize
        peak = np.max(np.abs(tr.data))
        if peak > 0:
            tr.data = tr.data / peak
            st_big += tr
            added = True

    if added:
        have.append(sta_str)
    else:
        nohave.append(sta_str)

print(f"Stations with data ({len(have)}): {have}")
print(f"Stations without data ({len(nohave)}): {nohave}")
print(st_big)

In [ ]:
#st_big.plot()


In [ ]:
#event_df

## 4. Build a single-event template example

Estimate a P arrival from epicentral distance and an assumed constant velocity, then extract a five-second waveform. This is an approximate window-placement method, not a measured P pick or a relocated source.


In [ ]:
client = Client("IRIS")

# Event information
ev_lat = event_df["latitude"].iloc[0]
ev_lon = event_df["longitude"].iloc[0]
origin_time = UTCDateTime(event_df["time"].iloc[0])

VP = 5.5                 # P-wave speed (km/s)
template_length = 5.0    # seconds

st_templates = Stream()
have, nohave = [], []

for sta_str in list_stations:   # "UW-ASR-EHZ"
    net, sta, cha = sta_str.split("-")

    # --- Get station coordinates ---
    try:
        inv = client.get_stations(network=net, station=sta, level="station")
        sta_lat = inv[0][0].latitude
        sta_lon = inv[0][0].longitude
    except:
        nohave.append(sta_str)
        continue

    # --- Distance event–station ---
    dist_m, _, _ = gps2dist_azimuth(ev_lat, ev_lon, sta_lat, sta_lon)
    dist_km = dist_m / 1000.0

    # --- Simple P arrival ---
    t_p = origin_time + (dist_km / VP)

    # --- We'll download from P−5 to P+10 seconds (safe window) ---
    t_start = t_p - 0.1
    t_end   = t_p + 10

    # --- Waveforms ---
    try:
        st = client.get_waveforms(net, sta, "*", cha, t_start, t_end)
    except:
        nohave.append(sta_str)
        continue

    if len(st) == 0:
        nohave.append(sta_str)
        continue

    st.merge(fill_value="interpolate")
    st.detrend("linear")

    # We only need the 5-s snippet after P
    for tr in st:
        tr.filter("bandpass", freqmin=2, freqmax=10, corners=4, zerophase=True)

        # Cut the 5-second template
        tr_temp = tr.copy().trim(t_p, t_p + template_length, pad=True, fill_value=0)

        # Normalize peak
        peak = np.max(np.abs(tr_temp.data))
        if peak == 0:
            continue
        tr_temp.data /= peak

        st_templates += tr_temp
        have.append(sta_str)

print("Templates created from stations:", have)
print("Stations with no data:", nohave)
print(st_templates)


In [ ]:
st_templates.plot()

## 5. Inspect predicted P/S windows

Plot station waveforms with approximate arrivals to check whether the window contains useful signal. These diagnostic filter/window choices are exploratory and should not be substituted for confirmed production settings.


In [ ]:
client = Client("IRIS")

# ======================================================
# Event info (from your event_df)
# ======================================================
ev_lat = float(event_df["latitude"].iloc[0])
ev_lon = float(event_df["longitude"].iloc[0])
origin_time = UTCDateTime(event_df["time"].iloc[0])

# Velocities
VP = 5.5     # km/s
VS = 3.2     # km/s

# Lists to track results
good_stations = []
failed_stations = []

# ======================================================
# Loop through every station in your list
# ======================================================
for sta_str in list_stations:
    net, sta, cha = sta_str.split("-")
    print(f"\n=== Processing {sta_str} ===")

    # -----------------------------
    # 1) Get station coordinates
    # -----------------------------
    try:
        inv = client.get_stations(network=net, station=sta, level="station")
        sta_lat = inv[0][0].latitude
        sta_lon = inv[0][0].longitude
    except:
        print("  -> No station metadata, skipping")
        failed_stations.append(sta_str)
        continue

    # -----------------------------
    # 2) Compute distance, P, S arrival
    # -----------------------------
    dist_m, az, baz = gps2dist_azimuth(ev_lat, ev_lon, sta_lat, sta_lon)
    dist_km = dist_m / 1000.0

    t_p = origin_time + (dist_km / VP)
    t_s = origin_time + (dist_km / VS)

    print(f"  Distance = {dist_km:.1f} km")
    print(f"  P arrival = {t_p}")
    print(f"  S arrival = {t_s}")

    # -----------------------------
    # 3) Download data around P–S
    # -----------------------------
    t_start = t_p - 0.5
    t_end   = t_start + 5

    try:
        st = client.get_waveforms(net, sta, "*", cha, t_start, t_end)
    except:
        print("  -> No waveform data, skipping")
        failed_stations.append(sta_str)
        continue

    if len(st) == 0:
        print("  -> Empty waveform, skipping")
        failed_stations.append(sta_str)
        continue

    tr = st[0]

    # Optional filter
    tr.filter("bandpass", freqmin=1, freqmax=10, corners=4, zerophase=True)

    # -----------------------------
    # 4) Plot with P and S arrival markers
    # -----------------------------
    times = tr.times(reftime=t_start)

    plt.figure(figsize=(14, 5))
    plt.plot(times, tr.data, linewidth=0.8)

    # P-wave marker
    plt.axvline((t_p - t_start), color='r', linestyle='--', label='P arrival')

    # S-wave marker
    plt.axvline((t_s - t_start), color='b', linestyle='--', label='S arrival')

    plt.title(f"{sta_str} | dist={dist_km:.1f} km")
    plt.xlabel("Time (s since window start)")
    plt.legend()
    plt.grid(True)
    plt.show()

    good_stations.append(sta_str)


# ======================================================
# Summary
# ======================================================
print("\n========================")
print("Stations plotted (have data):")
print(good_stations)

print("\nStations with no data:")
print(failed_stations)


## 6. Save the event-template collection

Build templates for catalog events, standardize sample counts at 50 Hz, and save HDF5 files with timing metadata. This downloads data and writes to the configured server directory; check paths before running.


In [ ]:
client = Client("IRIS")

# Base output path
base_path = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis"

# Velocity model (simple)
VP = 5.5  # km/s

# Template window relative to P
PRE_P  = 0.5   # seconds before P
POST_P = 4.5   # seconds after P
TEMPLATE_LENGTH = PRE_P + POST_P  # should be 5.0 s

# >>> target sampling rate for ALL templates
TARGET_FS = 50.0  # Hz

# ----------------------------------------------------
# (1) Pre-compute station coordinates (for all stations)
# ----------------------------------------------------
station_coords = {}  # key: "NET-STA-CHA" -> (lat, lon)

for sta_str in list_stations:   # e.g. "UW-ASR-EHZ"
    net, sta, cha = sta_str.split("-")
    try:
        inv = client.get_stations(network=net, station=sta, level="station")
        sta_lat = inv[0][0].latitude
        sta_lon = inv[0][0].longitude
        station_coords[sta_str] = (sta_lat, sta_lon)
    except Exception as e:
        print(f"[Station meta fail] {sta_str}: {e}")
        continue

print(f"Have coordinates for {len(station_coords)} stations")

# ----------------------------------------------------
# (2) Loop over events and create templates
# ----------------------------------------------------
for idx, row in event_df.iterrows():
    origin_time = UTCDateTime(row["time"])
    ev_lat = float(row["latitude"])
    ev_lon = float(row["longitude"])

    # Folder & event name based on origin time
    ev_name = origin_time.strftime("%Y%m%dT%H%M%S")
    ev_dir = os.path.join(base_path, ev_name)
    os.makedirs(ev_dir, exist_ok=True)

    h5_path = os.path.join(ev_dir, f"{ev_name}_templates.h5")
    print(f"\n=== Event {idx} | {ev_name} ===")
    print(f"Saving templates to: {h5_path}")

    with h5py.File(h5_path, "w") as h5:
        # store some event metadata
        h5.attrs["origin_time"] = str(origin_time)
        h5.attrs["event_latitude"] = ev_lat
        h5.attrs["event_longitude"] = ev_lon
        h5.attrs["target_sampling_rate"] = TARGET_FS
        
        n_saved = 0

        # Loop over stations with known coordinates
        for sta_str, (sta_lat, sta_lon) in station_coords.items():
            net, sta, cha = sta_str.split("-")

            # ---- Distance and P-arrival ----
            dist_m, _, _ = gps2dist_azimuth(ev_lat, ev_lon, sta_lat, sta_lon)
            dist_km = dist_m / 1000.0
            t_p = origin_time + (dist_km / VP)

            # Time window: 0.5 s before P, 4.5 s after P
            t_start = t_p - PRE_P
            t_end   = t_p + POST_P

            # Download waveform
            try:
                st = client.get_waveforms(net, sta, "*", cha, t_start, t_end)
            except Exception as e:
                print(f"  [No data] {sta_str}: {e}")
                continue

            if len(st) == 0:
                print(f"  [Empty] {sta_str}")
                continue

            st.merge(fill_value="interpolate")
            st.detrend("linear")

            # Use first trace for simplicity
            tr = st[0]

            # Bandpass filter (same as before)
            tr.filter("bandpass", freqmin=2, freqmax=10, corners=4, zerophase=True)

            # Trim exactly to desired window (in case of slight padding issues)
            tr_temp = tr.copy().trim(t_start, t_end, pad=True, fill_value=0)
            
                        # >>> Ensure common sampling rate (50 Hz)
            fs = tr_temp.stats.sampling_rate

            if abs(fs - 100.0) < 1e-6:
                # Downsample 100 Hz -> 50 Hz
                tr_temp.decimate(factor=2, no_filter=False)
                fs = tr_temp.stats.sampling_rate

            elif abs(fs - 50.0) < 1e-6:
                pass  # already fine
            else:
                print(f"  [Skip] {sta_str}: unexpected fs={fs} Hz")
                continue

            # Expected number of samples for 5.0 s at 50 Hz
            n_expected = int(round(TEMPLATE_LENGTH * TARGET_FS))  # 5.0 * 50 = 250
            n_actual = len(tr_temp.data)

            if n_actual > n_expected:
                # Too many samples (e.g. 251) -> crop
                tr_temp.data = tr_temp.data[:n_expected]
            elif n_actual < n_expected:
                # Too few -> pad with zeros (just in case)
                pad = np.zeros(n_expected - n_actual, dtype=tr_temp.data.dtype)
                tr_temp.data = np.concatenate([tr_temp.data, pad])
                print(f"  [Padded] {sta_str}: {n_actual} -> {n_expected} samples")

            # Optional: sanity print if you want
            # print(f"  {sta_str}: fs={fs}, samples={len(tr_temp.data)}")

            # Now guaranteed: len(tr_temp.data) == n_expected

            # Convert to float32 and normalize
            data = tr_temp.data.astype(np.float32)
            peak = np.max(np.abs(data))
            if peak == 0:
                print(f"  [Zero peak] {sta_str}")
                continue
            data /= peak

            # Create dataset, attrs etc. (unchanged)
            dname = sta_str
            dset = h5.create_dataset(dname, data=data, compression="gzip")
            dset.attrs["network"] = net
            dset.attrs["station"] = sta
            dset.attrs["channel"] = cha
            dset.attrs["sampling_rate"] = fs          # 50.0
            dset.attrs["distance_km"] = dist_km
            dset.attrs["p_pick_time_utc"] = str(t_p)
            dset.attrs["window_start_relative_to_P_sec"] = -PRE_P
            dset.attrs["window_end_relative_to_P_sec"] = POST_P
            dset.attrs["window_start_relative_to_origin_sec"] = float(t_start - origin_time)
            dset.attrs["window_end_relative_to_origin_sec"] = float(t_end - origin_time)

            n_saved += 1

## 7. Inspect saved templates

Read one saved event and plot its station waveforms. Check window alignment and missing stations before interpreting correlations.


In [ ]:
base_path = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis"
event_name = "20230827T101023"   # <-- cambia por la carpeta del evento que quieres ver

ev_dir = os.path.join(base_path, event_name)
h5_path = os.path.join(ev_dir, f"{event_name}_templates.h5")

print("reading file:", h5_path)

# ==========================
# Abrir el HDF5
# ==========================
with h5py.File(h5_path, "r") as h5:
    station_names = list(h5.keys())
    print(f"\nEstaciones en este evento ({len(station_names)}):")
    for name in station_names:
        print("  ", name)

    # Si no hay nada, salimos
    if len(station_names) == 0:
        print("No hay templates en este archivo.")
    else:
        # ==========================
        # Wiggle plot de todos los templates
        # ==========================
        plt.figure(figsize=(10, 0.4 * len(station_names)))

        for i, dname in enumerate(sorted(station_names)):
            dset = h5[dname]
            data = dset[:]                      # señal (ya normalizada)
            fs = dset.attrs["sampling_rate"]    # Hz

            # Vector de tiempo
            t = np.arange(len(data)) / fs

            # Tiempo relativo de la P (debería ser 0.5 s)
            # usamos el atributo para no hardcodear nada:
            t_start_rel_P = dset.attrs["window_start_relative_to_P_sec"]  # -0.5
            t_p_rel = -float(t_start_rel_P)    # entonces P está a 0.5 s

            # Offset vertical para no superponer todas las trazas
            offset = i * 2.0   # separa las trazas

            plt.plot(t, data + offset, linewidth=0.8, label=dname)

            # Nombre de estación a la derecha
            plt.text(t[-1] + 0.1, offset, dname, va="center", fontsize=8)

        # Línea vertical de P (misma para todos: 0.5 s)
        plt.axvline(t_p_rel, linestyle="--", linewidth=1.0)

        plt.xlabel("Tiempo (s desde el inicio de la ventana)")
        plt.ylabel("Estaciones (offset)")
        plt.title(f"Templates del evento {event_name} (P marcada con línea punteada)")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


## 8. Audit station coverage and sampling rates

Inspect the template files to identify inconsistent station coverage or sampling rates, which could otherwise alter stacked correlations.


In [ ]:
from collections import defaultdict

base_dir = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis/"

all_event_station_sets = []   # list of sets, one per event file
event_files = []              # to keep track of which file is which

# sampling rates seen per station across ALL events
sr_by_station = defaultdict(set)
stations_missing_sr = set()

# ==========================
# Loop over all subfolders
# ==========================
for folder in sorted(os.listdir(base_dir)):
    folder_path = os.path.join(base_dir, folder)
    if not os.path.isdir(folder_path):
        continue

    # Find templates HDF5 file in this folder
    h5_candidates = glob.glob(os.path.join(folder_path, "*templates*.h5"))
    if not h5_candidates:
        print(f"[{folder}] -> no templates .h5 file, skipping")
        continue

    h5_path = h5_candidates[0]  # assume one per folder
    print(f"\nLeyendo archivo: {h5_path}")

    try:
        with h5py.File(h5_path, "r") as h5:
            station_names = list(h5.keys())
            print(f"  Estaciones ({len(station_names)}): {station_names}")

            all_event_station_sets.append(set(station_names))
            event_files.append(h5_path)

            # -----------------------------
            # Check sampling rate per station
            # -----------------------------
            for sta in station_names:
                grp = h5[sta]
                sr = None

                # 1) Try as group attribute
                if "sampling_rate" in grp.attrs:
                    sr = float(grp.attrs["sampling_rate"])
                else:
                    # 2) Try in first dataset inside this group
                    ds_names = list(grp.keys())
                    if len(ds_names) > 0:
                        ds = grp[ds_names[0]]
                        if "sampling_rate" in ds.attrs:
                            sr = float(ds.attrs["sampling_rate"])
                        elif "delta" in ds.attrs:
                            # sampling_rate = 1 / delta
                            sr = 1.0 / float(ds.attrs["delta"])

                if sr is not None:
                    sr_by_station[sta].add(round(sr, 6))  # round a bit for float noise
                else:
                    stations_missing_sr.add(sta)

    except Exception as e:
        print(f"  ERROR abriendo {h5_path}: {e}")

# ==========================
# Compute intersection (common stations)
# ==========================
if not all_event_station_sets:
    print("\n⚠️ No se encontró ningún archivo de templates válido.")
else:
    common_stations = set.intersection(*all_event_station_sets)
    print("\n==========================================")
    print(f"Número de eventos (templates) leídos: {len(all_event_station_sets)}")
    print(f"Número de estaciones comunes a TODOS: {len(common_stations)}")
    print("Estaciones presentes en TODOS los templates:")
    print(sorted(common_stations))

# ==========================
# Sampling rate summary
# ==========================
print("\n==========================================")
print("Resumen de sampling rate por estación:")

for sta in sorted(sr_by_station.keys()):
    srs = sr_by_station[sta]
    if len(srs) == 1:
        print(f"  {sta}: fs = {list(srs)[0]} Hz")
    else:
        print(f"  ⚠ {sta}: múltiples sampling rates encontrados -> {srs}")

if stations_missing_sr:
    print("\n⚠ Estaciones donde no se pudo inferir sampling_rate (ni attr ni delta):")
    print(sorted(stations_missing_sr))


## 9. Inspect HDF5 contents

Examine a saved file before relying on its datasets and attributes. The following diagnostics depend on those saved templates.


In [ ]:

base_path = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis"

# pick one event folder (e.g. the first one)
folders = sorted([f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))])
print("Event folders:", folders[:5])  # just to see some

ev_name = folders[0]   # choose one, or set manually
h5_path = os.path.join(base_path, ev_name, f"{ev_name}_templates.h5")
print("Using file:", h5_path)

with h5py.File(h5_path, "r") as h5:
    for sta in h5.keys():
        dset = h5[sta]
        print(f"{sta}: shape={dset.shape}, fs={dset.attrs['sampling_rate']}")


## 10. Verify a template against its source waveform

Retrieve the same event waveform and compare it with the template at zero lag. A mismatch can indicate inconsistent window timing, filtering or sampling rather than a missing earthquake.


In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from obspy.signal.cross_correlation import correlate
import h5py
import numpy as np
import matplotlib.pyplot as plt

client = Client("IRIS")

TEMPLATE_ID = "20230827T101023"
TEMPLATE_DIR = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis"
h5_path = os.path.join(TEMPLATE_DIR, TEMPLATE_ID, f"{TEMPLATE_ID}_templates.h5")

sta_str = "UW-RCM-HHZ"   # pick one of your 11
net, sta, cha = sta_str.split("-")

with h5py.File(h5_path, "r") as h5:
    dset = h5[sta_str]
    temp_data = dset[()]                    # (250,)
    fs_temp = float(dset.attrs["sampling_rate"])
    p_time = UTCDateTime(dset.attrs["p_pick_time_utc"])

print("Template fs:", fs_temp)
print("P-pick time:", p_time)

# Fetch 1 minute around the P pick
t0 = p_time - 30
t1 = p_time + 30

st = client.get_waveforms(net, sta, "*", cha, t0, t1)
tr = st[0]
tr.detrend("linear")
tr.filter("bandpass", freqmin=2, freqmax=10, corners=4, zerophase=True)

# Resample if needed
if abs(tr.stats.sampling_rate - 100.0) < 1e-6:
    tr.decimate(factor=2, no_filter=False)

data = tr.data.astype(np.float32)
fs = tr.stats.sampling_rate

print("Continuous fs:", fs)

# Extract the same 5-s P window from continuous data
n_temp = len(temp_data)          # 250 samples → 5 s
i_start = int((p_time - 0.5 - t0) * fs)   # should match your t_start
i_end   = i_start + n_temp

snip = data[i_start:i_end]

print("Template length:", n_temp, "snip length:", len(snip))

# Normalize both
temp0 = temp_data - temp_data.mean()
snip0 = snip - snip.mean()
temp0 /= (np.std(temp0) + 1e-8)
snip0 /= (np.std(snip0) + 1e-8)

# Simple correlation using ObsPy (1D)
cc = correlate(snip0, temp0, 0)  # lag 0 only (they should already align)
print("1D CC at lag 0:", cc)

# Quick plot
t_axis = np.arange(n_temp) / fs
plt.figure()
plt.plot(t_axis, temp0, label="Template (norm)")
plt.plot(t_axis, snip0, label="Snippet around P (norm)", alpha=0.7)
plt.legend()
plt.xlabel("Time since P-window start (s)")
plt.show()


## 11. Repeat the alignment check using stored metadata

Use the recorded window offsets instead of a hardcoded start time to diagnose template/continuous-data alignment.


In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from obspy.signal.cross_correlation import correlate
import h5py
import numpy as np
import matplotlib.pyplot as plt

client = Client("IRIS")

TEMPLATE_ID = "20230827T101023"
TEMPLATE_DIR = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis"
h5_path = os.path.join(TEMPLATE_DIR, TEMPLATE_ID, f"{TEMPLATE_ID}_templates.h5")

sta_str = "UW-RCM-HHZ"   # you can also try CC-KAUT-BHZ etc.
net, sta, cha = sta_str.split("-")

with h5py.File(h5_path, "r") as h5:
    dset = h5[sta_str]
    temp_data = dset[()]                    # (250,)
    fs_temp = float(dset.attrs["sampling_rate"])
    p_time = UTCDateTime(dset.attrs["p_pick_time_utc"])
    t_start_rel = float(dset.attrs["window_start_relative_to_P_sec"])
    t_end_rel   = float(dset.attrs["window_end_relative_to_P_sec"])

print("Template fs:", fs_temp)
print("P-pick time:", p_time)
print("Template window rel to P:", t_start_rel, "to", t_end_rel)

# Fetch 1 minute around the P pick
t0 = p_time - 30
t1 = p_time + 30

st = client.get_waveforms(net, sta, "*", cha, t0, t1)
tr = st[0]
tr.detrend("linear")
tr.filter("bandpass", freqmin=2, freqmax=10, corners=4, zerophase=True)

# Resample if needed
if abs(tr.stats.sampling_rate - 100.0) < 1e-6:
    tr.decimate(factor=2, no_filter=False)

data = tr.data.astype(np.float32)
fs = tr.stats.sampling_rate
print("Continuous fs:", fs)

# Extract the same P-window from continuous data
n_temp = len(temp_data)          # 250 samples → 5 s
t_win_start = p_time + t_start_rel   # p_time - 0.5 s
offset_sec = t_win_start - t0
i_start = int(offset_sec * fs)
i_end   = i_start + n_temp

snip = data[i_start:i_end]
print("Template length:", n_temp, "snip length:", len(snip))

# Normalize both
temp0 = temp_data.astype(np.float64)
snip0 = snip.astype(np.float64)

temp0 -= temp0.mean()
snip0 -= snip0.mean()
temp0 /= (np.std(temp0) + 1e-8)
snip0 /= (np.std(snip0) + 1e-8)

# Simple correlation at zero lag
cc0 = correlate(snip0, temp0, 0)
print("1D CC at lag 0:", cc0)

# Quick plot
t_axis = np.arange(n_temp) / fs
plt.figure()
plt.plot(t_axis, temp0, label="Template (norm)")
plt.plot(t_axis, snip0, label="Continuous P-window (norm)", alpha=0.7)
plt.legend()
plt.xlabel("Time since window start (s)")
plt.title(sta_str)
plt.show()


## 12. Set up exploratory correlation comparisons

The following cells inspect selected traces, all-template station coverage, zero-lag correlation and window placement. They share and overwrite variables; run as diagnostics, not as a final batch pipeline.


In [ ]:
import os, glob, numpy as np
import matplotlib.pyplot as plt
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

# tus settings
COMMON_STATIONS = [
    "CC-KAUT-BHZ","CC-MILD-BHZ","CC-OBSR-BHZ","CC-PANH-BHZ",
    "CC-PARA-BHZ","CC-RUSH-BHZ","UW-FMW-HHZ","UW-LON-HHZ",
    "UW-RCM-HHZ","UW-RCS-EHZ","UW-RER-HHZ",
]
TARGET_FS = 50.0
LOW_CUT, HIGH_CUT = 2.0, 10.0
client = Client("IRIS")



In [ ]:
TEMPLATE_ID = "20230827T101023"
t_evt = UTCDateTime(TEMPLATE_ID)   # interpreta YYYYMMDDTHHMMSS como UTC
print("Event UTC:", t_evt.isoformat())


In [ ]:
import h5py, numpy as np
import matplotlib.pyplot as plt

TEMPLATE_H5 = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis/20230827T101023/20230827T101023_templates.h5"

show_stations = ["CC-KAUT-BHZ", "CC-MILD-BHZ", "UW-FMW-HHZ", "UW-RER-HHZ"]

with h5py.File(TEMPLATE_H5, "r") as h5:
    # sampling rate (take the first station that exists)
    fs = None
    for s in show_stations:
        if s in h5:
            fs = float(h5[s].attrs["sampling_rate"])
            break
    print("fs:", fs)

    fig, axes = plt.subplots(len(show_stations), 1, figsize=(9, 6), sharex=True)
    if len(show_stations) == 1:
        axes = [axes]

    for ax, sta in zip(axes, show_stations):
        if sta not in h5:
            ax.text(0.02, 0.7, f"{sta} NOT IN H5", transform=ax.transAxes)
            ax.axis("off")
            continue

        x = h5[sta][()].astype(float)
        t = np.arange(len(x)) / fs

        ax.plot(t, x, linewidth=0.8, color="0.2")
        ax.set_ylabel(sta, fontsize=9)
        ax.grid(alpha=0.25)

    axes[-1].set_xlabel("seconds in template (5 s window)")
    fig.suptitle("Template waveforms (raw)", y=0.98, fontsize=12)
    plt.tight_layout()
    plt.show()


In [ ]:
import h5py, numpy as np
import matplotlib.pyplot as plt

TEMPLATE_H5 = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis/20230827T101023/20230827T101023_templates.h5"

with h5py.File(TEMPLATE_H5, "r") as h5:
    stations = sorted(list(h5.keys()))
    fs = float(h5[stations[0]].attrs["sampling_rate"])

    n = len(stations)
    fig_h = max(6, 1.2*n)   # auto height
    fig, axes = plt.subplots(n, 1, figsize=(11, fig_h), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, sta in zip(axes, stations):
        x = h5[sta][()].astype(float)
        t = np.arange(len(x)) / fs

        # normalizar solo para visualizar (cada traza comparable)
        x = x - np.mean(x)
        x = x / (np.max(np.abs(x)) + 1e-12)

        ax.plot(t, x, color="0.2", linewidth=0.8)
        ax.set_ylabel(sta, fontsize=8)
        ax.grid(alpha=0.2)

    axes[-1].set_xlabel("seconds in template")
    fig.suptitle(f"Template: {TEMPLATE_H5}\nfs={fs} Hz, stations={n}", y=0.995, fontsize=12)
    plt.tight_layout()
    plt.show()


In [ ]:
import h5py
import numpy as np

TEMPLATE_H5 = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis/20230827T101023/20230827T101023_templates.h5"

with h5py.File(TEMPLATE_H5, "r") as h5:
    stations = sorted(list(h5.keys()))
    
    # tomar solo una estación primero (ej: KAUT)
    sta = stations[0]
    x = h5[sta][()].astype(float)

# normalizar igual que en correlate()
norm = np.linalg.norm(x)
cc_self = np.dot(x, x) / (norm * norm)

print("Self correlation:", cc_self)


In [ ]:
import numpy as np, h5py
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

client = Client("IRIS")

TEMPLATE_H5 = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis/20230827T101023/20230827T101023_templates.h5"
TEMPLATE_ID = "20230827T101023"
STA = "CC-KAUT-BHZ"   # usa la misma que graficaste
LOW_CUT, HIGH_CUT = 2.0, 10.0
FS_TARGET = 50.0

t_evt = UTCDateTime(TEMPLATE_ID)

# template
with h5py.File(TEMPLATE_H5, "r") as h5:
    tpl = h5[STA][()].astype(float)
    fs_tpl = float(h5[STA].attrs["sampling_rate"])
print("fs_tpl:", fs_tpl, "n_tpl:", len(tpl))

# continuo 10 s alrededor del evento
net, sta, cha = STA.split("-")
t0 = t_evt - 5
t1 = t_evt + 5
tr = client.get_waveforms(net, sta, "*", cha, t0, t1)[0]
tr.detrend("linear")

if abs(tr.stats.sampling_rate - 100.0) < 1e-6:
    tr.decimate(2, no_filter=False)

tr.filter("bandpass", freqmin=LOW_CUT, freqmax=HIGH_CUT, corners=4, zerophase=True)

x = tr.data.astype(float)
fs = tr.stats.sampling_rate
print("fs_cont:", fs, "n_cont:", len(x))



In [ ]:
import numpy as np, h5py
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

client = Client("IRIS")

TEMPLATE_H5 = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis/20230827T101023/20230827T101023_templates.h5"
TEMPLATE_ID = "20230827T101023"
t_evt = UTCDateTime(TEMPLATE_ID)

LOW_CUT, HIGH_CUT = 2.0, 10.0
FS_TARGET = 50.0

def norm(v):
    v = v.astype(float)
    v = v - np.mean(v)
    return v / (np.linalg.norm(v) + 1e-12)

# load list of stations in the H5 (this is the truth of what your template contains)
with h5py.File(TEMPLATE_H5, "r") as h5:
    stations = sorted(list(h5.keys()))
    fs_tpl = float(h5[stations[0]].attrs["sampling_rate"])
print("Stations in template:", len(stations))
print("Template fs:", fs_tpl)
print("Event UTC:", t_evt.isoformat())

# 10 s buffer around event
t0 = t_evt - 5
t1 = t_evt + 5

results = []
best_cc_list = []
best_shift_list = []

for STA in stations:
    # --- load template trace for this station ---
    with h5py.File(TEMPLATE_H5, "r") as h5:
        tpl = h5[STA][()].astype(float)
        fs_this = float(h5[STA].attrs["sampling_rate"])

    if abs(fs_this - fs_tpl) > 1e-6:
        print("[skip] fs mismatch in template for", STA, fs_this)
        continue

    tpln = norm(tpl)
    nt = len(tpln)

    # --- fetch continuous around event for this station ---
    net, sta, cha = STA.split("-")
    try:
        tr = client.get_waveforms(net, sta, "*", cha, t0, t1)[0]
    except Exception as e:
        results.append((STA, np.nan, np.nan, f"no data: {e}"))
        continue

    tr.detrend("linear")

    # decimate to 50 Hz if needed (match your pipeline)
    if abs(tr.stats.sampling_rate - 100.0) < 1e-6:
        tr.decimate(2, no_filter=False)

    # must be 50 Hz now
    fs = tr.stats.sampling_rate
    if abs(fs - FS_TARGET) > 1e-6:
        results.append((STA, np.nan, np.nan, f"unexpected fs={fs}"))
        continue

    # same bandpass
    tr.filter("bandpass", freqmin=LOW_CUT, freqmax=HIGH_CUT, corners=4, zerophase=True)
    x = tr.data.astype(float)

    if len(x) < nt:
        results.append((STA, np.nan, np.nan, "buffer shorter than template"))
        continue

    # --- CC at lag0 (centered on event) ---
    center = int(round(5 * fs))           # because buffer is [-5,+5]
    start0 = center - nt//2
    if start0 < 0 or start0 + nt > len(x):
        results.append((STA, np.nan, np.nan, "template doesn't fit centered"))
        continue
    cc0 = float(np.dot(tpln, norm(x[start0:start0+nt])))

    # --- Best CC over lag (slide template over the 10 s buffer) ---
    best = -1.0
    best_i = None
    for i in range(0, len(x) - nt + 1):
        cc = float(np.dot(tpln, norm(x[i:i+nt])))
        if cc > best:
            best = cc
            best_i = i

    best_shift_sec = (best_i - start0) / fs  # relative to lag0-center
    results.append((STA, cc0, best, best_shift_sec))

    best_cc_list.append(best)
    best_shift_list.append(best_shift_sec)

# ---------------------------
# Print results sorted by Best CC
# ---------------------------
results_sorted = sorted(results, key=lambda r: (np.nan_to_num(r[2], nan=-1e9)), reverse=True)

print("\nPer-station results (sorted by Best CC):")
for STA, cc0, best, shift in results_sorted:
    if isinstance(shift, str):
        print(f"{STA:12s}  cc_lag0={cc0!s:>8}  best={best!s:>8}  shift={shift}")
    else:
        print(f"{STA:12s}  cc_lag0={cc0:8.3f}  best={best:8.3f}  shift={shift:7.2f} s")

# ---------------------------
# Network-style summary
# ---------------------------
best_cc_arr = np.array([b for b in best_cc_list if np.isfinite(b)])
shift_arr   = np.array([s for s in best_shift_list if np.isfinite(s)])

print("\nNetwork summary (using per-station BEST over lag):")
print("  mean(best_cc) =", float(np.mean(best_cc_arr)) if len(best_cc_arr) else np.nan)
print("  median(best_cc) =", float(np.median(best_cc_arr)) if len(best_cc_arr) else np.nan)
print("  min/max(best_cc) =", (float(np.min(best_cc_arr)), float(np.max(best_cc_arr))) if len(best_cc_arr) else (np.nan, np.nan))

print("\nShift summary (sec, relative to lag0-centered window):")
print("  mean shift =", float(np.mean(shift_arr)) if len(shift_arr) else np.nan)
print("  median shift =", float(np.median(shift_arr)) if len(shift_arr) else np.nan)
print("  min/max shift =", (float(np.min(shift_arr)), float(np.max(shift_arr))) if len(shift_arr) else (np.nan, np.nan))



In [ ]:
import numpy as np, h5py
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

client = Client("IRIS")

TEMPLATE_H5 = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis/20230827T101023/20230827T101023_templates.h5"
TEMPLATE_ID = "20230827T101023"
STA = "UW-RCM-HHZ"   # estación fuerte (~0.458)

LOW_CUT, HIGH_CUT = 2.0, 10.0
FS_TARGET = 50.0

def norm(v):
    v = v.astype(float)
    v = v - np.mean(v)
    return v / (np.linalg.norm(v) + 1e-12)

t_evt = UTCDateTime(TEMPLATE_ID)

# --- load template ---
with h5py.File(TEMPLATE_H5, "r") as h5:
    tpl = h5[STA][()].astype(float)
    fs = float(h5[STA].attrs["sampling_rate"])

tpln = norm(tpl)
nt = len(tpln)

# --- fetch 10 s around event ---
net, sta, cha = STA.split("-")
t0 = t_evt - 5
t1 = t_evt + 5

tr = client.get_waveforms(net, sta, "*", cha, t0, t1)[0]
tr.detrend("linear")

if abs(tr.stats.sampling_rate - 100.0) < 1e-6:
    tr.decimate(2, no_filter=False)

tr.filter("bandpass", freqmin=LOW_CUT, freqmax=HIGH_CUT, corners=4, zerophase=True)
x = tr.data.astype(float)

# --- find best alignment ---
best = -1.0
best_i = None
for i in range(0, len(x) - nt + 1):
    cc = float(np.dot(tpln, norm(x[i:i+nt])))
    if cc > best:
        best = cc
        best_i = i

seg_best = norm(x[best_i:best_i+nt])
print("Original best CC (full template):", best)


## 13. Test the effect of shortening a template

Remove the first two seconds to investigate the preceding correlation diagnostic. This experiment does not redefine the five-second production template length.


In [ ]:
# cortar primeros 2 segundos del template
cut_seconds = 2.0
cut_samples = int(cut_seconds * fs)

tpl_cut = tpl[cut_samples:]
tpl_cut = norm(tpl_cut)
nt_cut = len(tpl_cut)

# tomar mismo segmento alineado pero recortado
seg_cut = norm(x[best_i+cut_samples : best_i+cut_samples+nt_cut])

cc_cut = float(np.dot(tpl_cut, seg_cut))
print("CC after removing first 2 seconds:", cc_cut)


## 14. Inspect the four-West example subset

These example cells explicitly use four stations. They are retained for provenance and must not be relabeled as a five-station production run.


In [ ]:
TEMPLATE_H5 = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis/20230827T101023/20230827T101023_templates.h5"

WEST_STATIONS = [
    "CC-KAUT-BHZ",
    "UW-LON-HHZ",
    "CC-MILD-BHZ",
    "CC-PARA-BHZ",
]

# --- load template traces for west stations ---
with h5py.File(TEMPLATE_H5, "r") as h5:
    available = list(h5.keys())
    use = [s for s in WEST_STATIONS if s in available]
    missing = [s for s in WEST_STATIONS if s not in available]

    if not use:
        raise ValueError(f"None of WEST_STATIONS found in {TEMPLATE_H5}.\nAvailable: {available}")

    # read traces
    traces = []
    fs = None
    for sta in use:
        dset = h5[sta]
        x = dset[()].astype(float)
        traces.append(x)

        this_fs = float(dset.attrs.get("sampling_rate", np.nan))
        if fs is None:
            fs = this_fs
        elif abs(fs - this_fs) > 1e-6:
            print(f"[warn] sampling rate mismatch: {sta} has {this_fs}, others {fs}")

traces = np.array(traces)  # shape: (n_sta, nt)
nt = traces.shape[1]
t = np.arange(nt) / fs

# --- plot (stacked) ---
fig, axes = plt.subplots(len(use), 1, figsize=(12, 1.8*len(use)), sharex=True)

if len(use) == 1:
    axes = [axes]

for ax, sta, x in zip(axes, use, traces):
    # normalize for display only (keeps shapes comparable)
    x_disp = x / (np.max(np.abs(x)) + 1e-12)
    ax.plot(t, x_disp, linewidth=1.2)
    ax.set_ylabel(sta, rotation=0, labelpad=50, va="center")
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("seconds in template (5 s window)")
fig.suptitle(f"Template waveforms (WEST stations only)  |  fs={fs:.1f} Hz", y=0.98)
plt.tight_layout()
plt.show()

print("Using stations:", use)
if missing:
    print("Missing stations (not in H5):", missing)


## 15. Test correlation over one hour

Download continuous data for the example subset and compare sliding-window approaches in the next cells. The purpose is to inspect arrival alignment and correlation behavior, not to create the final three-month catalog.


In [ ]:
import numpy as np, h5py
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
import matplotlib.pyplot as plt

client = Client("IRIS")

TEMPLATE_ID = "20230827T101023"
TEMPLATE_H5 = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis/20230827T101023/20230827T101023_templates.h5"
WEST_STATIONS = ["CC-KAUT-BHZ","UW-LON-HHZ","CC-MILD-BHZ","CC-PARA-BHZ"]

LOW_CUT, HIGH_CUT = 2.0, 10.0
FS = 50.0
TEMPLATE_SECONDS = 5.0
nt = int(TEMPLATE_SECONDS * FS)

t_hour0 = UTCDateTime("2023-08-27T10:00:00")
t_hour1 = UTCDateTime("2023-08-27T11:00:00")

STEP_SEC = 5
step = int(STEP_SEC * FS)          # 10 samples
LAG_SEC = 1
lag = int(LAG_SEC * FS)            # 125 samples

def sliding_norm(x, w):
    # sqrt(sum(x^2)) in a sliding window
    x2 = x*x
    c = np.cumsum(np.r_[0.0, x2])
    s = c[w:] - c[:-w]
    return np.sqrt(s) + 1e-12

def norm_xcorr_fft(x, t):
    """
    Normalized cross-correlation of template t (len nt) with signal x (len N),
    producing CC for each start index (valid positions): len = N-nt+1.
    Uses FFT for numerator + sliding energy for denominator.
    """
    x = x.astype(float)
    t = t.astype(float)

    # zero-mean both
    t0 = t - t.mean()
    x0 = x - x.mean()  # good enough for 5s windows; if you want exact window-mean, it's slower

    # FFT-based correlation numerator (valid)
    n = len(x0) + len(t0) - 1
    nfft = 1 << (n - 1).bit_length()

    X = np.fft.rfft(x0, nfft)
    T = np.fft.rfft(t0[::-1], nfft)
    corr_full = np.fft.irfft(X * T, nfft)[:n]

    # "valid" part corresponds to aligning t within x
    # start indices 0..N-nt
    num = corr_full[nt-1:nt-1 + (len(x0)-nt+1)]

    # denominator: ||t|| * ||x_window||
    den = (np.linalg.norm(t0) + 1e-12) * sliding_norm(x0, nt)

    return num / den  # length N-nt+1

# 1) load templates
tpl = {}
with h5py.File(TEMPLATE_H5, "r") as h5:
    for sta in WEST_STATIONS:
        tpl[sta] = h5[sta][()].astype(float)

# 2) fetch continuous and compute per-station CC series
cc_series = []
Nmin = None

for sta_str in WEST_STATIONS:
    net, sta, cha = sta_str.split("-")
    tr = client.get_waveforms(net, sta, "*", cha, t_hour0, t_hour1)[0]
    tr.detrend("linear")
    if abs(tr.stats.sampling_rate - 100.0) < 1e-6:
        tr.decimate(2, no_filter=False)
    tr.filter("bandpass", freqmin=LOW_CUT, freqmax=HIGH_CUT, corners=4, zerophase=True)

    x = tr.data.astype(float)
    cc = norm_xcorr_fft(x, tpl[sta_str])

    # keep same length across stations (trim to shortest)
    if Nmin is None:
        Nmin = len(cc)
    else:
        Nmin = min(Nmin, len(cc))
    cc_series.append(cc)

cc_series = [c[:Nmin] for c in cc_series]
CC = np.vstack(cc_series)  # shape (nsta, Nmin)

# 3) for each time sample (start index), take max within ±lag (per station)
# pad and use sliding max (simple loop over stations; still fast)
CC_best = np.empty((CC.shape[0], Nmin), float)
for j in range(CC.shape[0]):
    c = CC[j]
    padded = np.pad(c, (lag, lag), mode="edge")
    # sliding window max of width 2*lag+1
    # brute but vectorized via stacking indices:
    w = 2*lag + 1
    # use stride trick
    shape = (Nmin, w)
    strides = (padded.strides[0], padded.strides[0])
    windows = np.lib.stride_tricks.as_strided(padded, shape=shape, strides=strides)
    CC_best[j] = windows.max(axis=1)

# 4) sample every "step" to match your STEP_SEC grid and stack stations
idx = np.arange(0, Nmin, step)
cc_stack = CC_best[:, idx].mean(axis=0)

t_mid = (idx + nt/2) / FS  # seconds since t_hour0

imax = int(np.nanargmax(cc_stack))
t_peak = (t_hour0 + float(t_mid[imax])).datetime

plt.figure(figsize=(12,4))
plt.plot(t_mid/60.0, cc_stack, linewidth=1.0)
plt.axvline(t_mid[imax]/60.0, linestyle="--", color="gray", linewidth=1.0)
plt.xlabel("Minutes since 10:00")
plt.ylabel("CC (mean of per-station best)")
plt.title(f"Detection (FAST FFT) | step={STEP_SEC}s, lag=±{LAG_SEC}s")
plt.grid(alpha=0.3)
plt.show()

print("Max CC:", float(cc_stack[imax]))
print("Peak time UTC:", t_peak)


In [ ]:
import numpy as np, h5py
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
import matplotlib.pyplot as plt

client = Client("IRIS")

TEMPLATE_ID = "20230827T101023"
TEMPLATE_H5 = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis/20230827T101023/20230827T101023_templates.h5"
WEST_STATIONS = ["CC-KAUT-BHZ","UW-LON-HHZ","CC-MILD-BHZ","CC-PARA-BHZ"]

LOW_CUT, HIGH_CUT = 2.0, 10.0
FS = 50.0
TEMPLATE_SECONDS = 5.0
nt = int(TEMPLATE_SECONDS * FS)

t0 = UTCDateTime("2023-08-27T10:00:00")
t1 = UTCDateTime("2023-08-27T11:00:00")

STEP_SEC = 5
step = int(STEP_SEC * FS)

LAG_SEC = 2.5
lag = int(LAG_SEC * FS)

def sliding_norm(x, w):
    x2 = x*x
    c = np.cumsum(np.r_[0.0, x2])
    s = c[w:] - c[:-w]
    return np.sqrt(s) + 1e-12

def norm_xcorr_valid(x, t):
    """
    Normalized CC for each window start (valid), without shifting the window.
    Returns c[t] = corr(template, x_window) at ZERO-LAG alignment (template starts at window start).
    """
    x = x.astype(float)
    t = t.astype(float)
    t0 = t - t.mean()
    x0 = x - x.mean()

    n = len(x0) + len(t0) - 1
    nfft = 1 << (n - 1).bit_length()

    X = np.fft.rfft(x0, nfft)
    T = np.fft.rfft(t0[::-1], nfft)
    corr_full = np.fft.irfft(X * T, nfft)[:n]

    # valid numerator: for window start i, correlation at zero-lag corresponds to index nt-1+i
    num = corr_full[nt-1:nt-1 + (len(x0)-nt+1)]
    den = (np.linalg.norm(t0) + 1e-12) * sliding_norm(x0, nt)
    return num / den  # length = N-nt+1

# 1) load templates
tpl = {}
with h5py.File(TEMPLATE_H5, "r") as h5:
    for sta in WEST_STATIONS:
        tpl[sta] = h5[sta][()].astype(float)

# 2) fetch continuous and compute per-station zero-lag CC time series
CC0_list = []
Nmin = None

for sta_str in WEST_STATIONS:
    net, sta, cha = sta_str.split("-")
    tr = client.get_waveforms(net, sta, "*", cha, t0, t1)[0]
    tr.detrend("linear")
    if abs(tr.stats.sampling_rate - 100.0) < 1e-6:
        tr.decimate(2, no_filter=False)
    tr.filter("bandpass", freqmin=LOW_CUT, freqmax=HIGH_CUT, corners=4, zerophase=True)

    x = tr.data.astype(float)
    cc0 = norm_xcorr_valid(x, tpl[sta_str])  # zero-lag CC for each start index

    if Nmin is None:
        Nmin = len(cc0)
    else:
        Nmin = min(Nmin, len(cc0))
    CC0_list.append(cc0)

CC0 = np.vstack([c[:Nmin] for c in CC0_list])  # (nsta, Nmin)

# 3) for each fixed window start, take MAX over LAGS by looking around the zero-lag index
# IMPORTANT: this is lag search INSIDE the correlation function, NOT shifting the window in time.
# Here, "lag" in samples means checking cc0 at nearby indices.
# This is equivalent to allowing +/-lag alignment inside the 5 s window.
idx = np.arange(lag, Nmin - lag, step)
t_mid = (idx + nt/2) / FS  # seconds since t0

cc_sta_best = []
for j in range(CC0.shape[0]):
    # for each station, for each window start idx, take max in [idx-lag : idx+lag]
    w = 2*lag + 1
    padded = np.pad(CC0[j], (lag, lag), mode="edge")
    win = np.lib.stride_tricks.as_strided(
        padded, shape=(Nmin, w),
        strides=(padded.strides[0], padded.strides[0])
    )
    best = win[:, :].max(axis=1)  # best at each index
    cc_sta_best.append(best[idx])

cc_sta_best = np.vstack(cc_sta_best)  # (nsta, nwin)
cc_stack = cc_sta_best.mean(axis=0)

# 4) plot
imax = int(np.nanargmax(cc_stack))
t_peak = (t0 + float(t_mid[imax])).datetime

plt.figure(figsize=(12,4))
plt.plot(t_mid/60.0, cc_stack, linewidth=1.0)
plt.axvline(t_mid[imax]/60.0, linestyle="--", color="gray", linewidth=1.0)
plt.xlabel(f"Minutes since {t0.strftime('%H:%M')}")
plt.ylabel("CC (mean over stations)")
plt.title(f"DAS-like: fixed windows + max over lag (±{LAG_SEC}s) | step={STEP_SEC}s")
plt.grid(alpha=0.3)
plt.show()

print("Max CC:", float(cc_stack[imax]))
print("Peak time UTC:", t_peak)


In [ ]:
import numpy as np, h5py
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
import matplotlib.pyplot as plt

client = Client("IRIS")

# -----------------------
# SETTINGS
# -----------------------
TEMPLATE_ID = "20230827T101023"
TEMPLATE_H5 = "/data/data4/veronica-scratch-rainier/swarm_august2023/templates-files/seis/20230827T101023/20230827T101023_templates.h5"
WEST_STATIONS = ["CC-KAUT-BHZ","UW-LON-HHZ","CC-MILD-BHZ","CC-PARA-BHZ"]

LOW_CUT, HIGH_CUT = 2.0, 10.0
FS = 50.0
TEMPLATE_SECONDS = 5.0
nt = int(TEMPLATE_SECONDS * FS)

t0 = UTCDateTime("2023-08-27T9:00:00")
t1 = UTCDateTime("2023-08-27T11:00:00")

STEP_SEC = 5
step = int(STEP_SEC * FS)

LAG_SEC = 0
lag = int(LAG_SEC * FS)

# -----------------------
# FAST sliding helpers
# -----------------------
def sliding_sum(x, w):
    c = np.cumsum(np.r_[0.0, x])
    return (c[w:] - c[:-w])

def norm_xcorr_valid_windowmean(x, t):
    """
    Proper normalized CC in [-1, 1] for each valid start:
      cc[i] = < (x_win - mean(x_win)), (t - mean(t)) > / ( ||x0_win|| * ||t0|| )
    Returns length N-nt+1.
    """
    x = x.astype(float)
    t = t.astype(float)

    nt = len(t)
    N = len(x)

    # template zero-mean + norm
    t0 = t - t.mean()
    t_norm = np.linalg.norm(t0) + 1e-12

    # sliding window mean for x
    sum_x  = sliding_sum(x, nt)
    sum_x2 = sliding_sum(x*x, nt)
    mean_x = sum_x / nt

    # ||x0_window|| = sqrt( sum((x-mean)^2) ) = sqrt( sum_x2 - nt*mean^2 )
    energy0 = np.maximum(sum_x2 - nt*(mean_x**2), 0.0)
    x0_norm = np.sqrt(energy0) + 1e-12

    # FFT correlation numerator: sum( x * t0_shifted )  (valid)
    n = N + nt - 1
    nfft = 1 << (n - 1).bit_length()
    X = np.fft.rfft(x, nfft)
    T = np.fft.rfft(t0[::-1], nfft)
    corr_full = np.fft.irfft(X * T, nfft)[:n]
    num = corr_full[nt-1:nt-1 + (N-nt+1)]

    # IMPORTANT: because t0 is zero-mean, <mean_x, t0> = 0, so num is already correct
    return num / (t_norm * x0_norm)

# -----------------------
# 1) Load template traces
# -----------------------
tpl = {}
with h5py.File(TEMPLATE_H5, "r") as h5:
    for sta in WEST_STATIONS:
        tpl[sta] = h5[sta][()].astype(float)

print("Template shape:", (len(next(iter(tpl.values()))), len(tpl)))

# -----------------------
# 2) Fetch continuous + compute CC0 per station
# -----------------------
CC0_list = []
Nmin = None

for sta_str in WEST_STATIONS:
    net, sta, cha = sta_str.split("-")
    tr = client.get_waveforms(net, sta, "*", cha, t0, t1)[0]

    tr.detrend("linear")
    if abs(tr.stats.sampling_rate - 100.0) < 1e-6:
        tr.decimate(2, no_filter=False)  # -> 50 Hz

    tr.filter("bandpass", freqmin=LOW_CUT, freqmax=HIGH_CUT, corners=4, zerophase=True)

    x = tr.data.astype(float)
    cc0 = norm_xcorr_valid_windowmean(x, tpl[sta_str])  # length N-nt+1

    if Nmin is None:
        Nmin = len(cc0)
    else:
        Nmin = min(Nmin, len(cc0))

    CC0_list.append(cc0)

CC0 = np.vstack([c[:Nmin] for c in CC0_list])  # (nsta, Nstarts)
print("CC0 matrix:", CC0.shape, "(nsta, Nstarts)")

# sanity check: normalized CC must be within [-1, 1]
print("Sanity: CC0 min/max =", float(np.nanmin(CC0)), float(np.nanmax(CC0)))

# -----------------------
# 3) DAS-like: fixed windows + max over lag in CC-space
# -----------------------
idx = np.arange(lag, Nmin - lag, step)
t_mid = (idx + nt/2) / FS  # seconds since t0

cc_sta_best = np.empty((CC0.shape[0], len(idx)), float)

for j in range(CC0.shape[0]):
    c = CC0[j]
    padded = np.pad(c, (lag, lag), mode="edge")
    w = 2*lag + 1
    windows = np.lib.stride_tricks.as_strided(
        padded,
        shape=(Nmin, w),
        strides=(padded.strides[0], padded.strides[0]),
    )
    best_all = windows.max(axis=1)
    cc_sta_best[j] = best_all[idx]

cc_stack = cc_sta_best.mean(axis=0)

# -----------------------
# 4) Plot + peak
# -----------------------
imax = int(np.nanargmax(cc_stack))
t_peak = (t0 + float(t_mid[imax])).datetime

plt.figure(figsize=(12,4))
plt.plot(t_mid/60.0, cc_stack, linewidth=1.0)
plt.axvline(t_mid[imax]/60.0, linestyle="--", color="gray", linewidth=1.0)
plt.xlabel(f"Minutes since {t0.strftime('%H:%M')}")
plt.ylabel("CC (mean over stations)")
plt.title(f"DAS-like (proper norm): fixed windows + max over lag (±{LAG_SEC}s) | step={STEP_SEC}s")
plt.grid(alpha=0.3)
plt.show()

print("Max CC:", float(cc_stack[imax]))
print("Peak time UTC:", t_peak)
print("Baseline (median CC):", float(np.median(cc_stack)))


## 16. Inspect saved three-month correlation results

Load previously computed correlation arrays, reconstruct UTC times and plot candidate peaks. Only load trusted local NPY files because this cell enables pickle. The displayed fixed threshold is diagnostic, not the manuscript MAD criterion.


In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from obspy import UTCDateTime

# ----------------------------
# 1) point to the folder with cc_*.npy
# ----------------------------
cc_dir = "/data/data4/veronica-scratch-rainier/swarm_august2023/results_cc_seis_test_noisefloor/CC_5sec-tem_2023-08-23_21.00-2023-11-23_21.00/20230827T101023"
pattern = os.path.join(cc_dir, "cc_*.npy")

files = sorted(glob.glob(pattern))
print("Found", len(files), "files")
if not files:
    raise RuntimeError(f"No cc_*.npy found in: {cc_dir}")

# ----------------------------
# 2) load + concatenate
# ----------------------------
t_all = []
cc_all = []

for f in files:
    d = np.load(f, allow_pickle=True).item()

    # keys from your writer:
    # "t_block_start_utc" (ISO string), "times_sec" (relative), "cc_windows" (values)
    t_block = UTCDateTime(d["t_block_start_utc"])
    t_rel = np.asarray(d["times_sec"], dtype=float)
    cc = np.asarray(d["cc_windows"], dtype=float)

    # absolute time for each window
    t_abs = np.array([(t_block + float(s)).datetime for s in t_rel], dtype=object)

    t_all.append(t_abs)
    cc_all.append(cc)

t_all = np.concatenate(t_all)
cc_all = np.concatenate(cc_all)

# sort by time
order = np.argsort(t_all)
t_all = t_all[order]
cc_all = cc_all[order]

print("Total windows:", len(cc_all))
print("CC min/median/max:", float(np.nanmin(cc_all)), float(np.nanmedian(cc_all)), float(np.nanmax(cc_all)))

# ----------------------------
# 3) plotting parameters
# ----------------------------
THRESH = 0.55  # adjust
hits = (cc_all >= THRESH) & np.isfinite(cc_all)

out_png = os.path.join(cc_dir, "cc_vs_time.png")

# ----------------------------
# 4) professional plot
# ----------------------------
plt.figure(figsize=(13, 4))

# main series: thin gray line
plt.plot(
    t_all, cc_all,
    lw=0.8,
    color="0.35",
    label="CC (stack)"
)

# threshold line
plt.axhline(THRESH, ls="--", lw=1.2, color="0.6", label=f"threshold = {THRESH:g}")

# hits
plt.scatter(
    np.array(t_all, dtype=object)[hits],
    cc_all[hits],
    s=18,
    color="black",
    zorder=3,
    label=f"hits (n={int(hits.sum())})"
)

# cosmetics
plt.ylim(-0.05, 1.05)
plt.xlabel("UTC time")
plt.ylabel("Cross-correlation (CC)")
plt.title("Template 20230827T101023 – CC vs time (seismic network)")
plt.grid(True, which="major", alpha=0.25)
plt.grid(True, which="minor", alpha=0.10)
plt.minorticks_on()
plt.legend(frameon=False, loc="upper right")
plt.gcf().autofmt_xdate()

plt.tight_layout()
plt.savefig(out_png, dpi=200)
plt.show()
plt.ylim()

print("Saved:", out_png)


## 17. Adjust the diagnostic plot

Reuse the previously loaded arrays and hit mask to inspect the correlation time series.


In [ ]:
plt.figure(figsize=(13, 4))

# main series: thin gray line
plt.plot(
    t_all, cc_all,
    lw=0.8,
    color="0.35",
    label="CC (stack)"
)

# threshold line
plt.axhline(THRESH, ls="--", lw=1.2, color="0.6", label=f"threshold = {THRESH:g}")

# hits
plt.scatter(
    np.array(t_all, dtype=object)[hits],
    cc_all[hits],
    s=18,
    color="black",
    zorder=3,
    label=f"hits (n={int(hits.sum())})"
)

# cosmetics
plt.ylim(-0.05, 1.05)
plt.xlabel("UTC time")
plt.ylabel("Cross-correlation (CC)")
plt.title("Template 20230827T101023 – CC vs time (seismic network)")
plt.grid(True, which="major", alpha=0.25)
plt.grid(True, which="minor", alpha=0.10)
plt.minorticks_on()
plt.legend(frameon=False, loc="upper right")
plt.gcf().autofmt_xdate()

plt.tight_layout()
plt.ylim(0.1,0.7)
plt.show()

## 18. Export diagnostic hits: review caveat

This cell changes THRESH to 0.4 but reuses the existing hits mask without recomputing it. Its printed threshold can therefore disagree with the selected rows. Code is preserved for review; do not treat hits_cc.csv as a final catalog.


In [ ]:
import csv
THRESH = 0.4

# --- list hits (time + CC) ---
hit_idx = np.where(hits)[0]

print(f"\nHits above threshold (CC >= {THRESH:g}): {len(hit_idx)}\n")

# sort hits by CC descending (or comment this out to keep chronological order)
hit_idx = hit_idx[np.argsort(cc_all[hit_idx])[::-1]]

for k, i in enumerate(hit_idx, 1):
    print(f"{k:02d}) {t_all[i]}   CC = {cc_all[i]:.6f}")

# --- optional: also write CSV next to the plot ---
out_csv = os.path.join(cc_dir, "hits_cc.csv")
with open(out_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["rank", "time_utc", "cc"])
    for k, i in enumerate(hit_idx, 1):
        w.writerow([k, str(t_all[i]), float(cc_all[i])])

print("\nSaved hits table:", out_csv)


## 19. Load comparison libraries

Prepare the saved-catalog comparison. It does not rerun template matching.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from obspy import UTCDateTime

## 20. Compare saved DAS and station detections

Read the existing result CSVs, normalize their time columns, and perform a greedy one-to-one comparison within the configured tolerance. Confirm file provenance and clock treatment before interpreting overlap.


In [ ]:
import matplotlib.dates as mdates

# =============================
# INPUTS
# =============================
DAS_CSV  = "/data/data2/veronica-rainier/cc_results/CC_5sec-tem_2023-08-23_21.00-2023-11-23_21.00/mad20/DAS_global_dedup_byMinute_keepMax_cc004.csv"
SEIS_CSV = "/data/data4/veronica-scratch-rainier/swarm_august2023/results_cc_seis_test_noisefloor/CC_5sec-tem_2023-08-23_21.00-2023-11-23_21.00/detecciones_MAD_15/detecciones_global_dedup_6s.csv"
TOL_SEC  = 10

# Swarm shading window (edit if you want)
SWARM_START = "2023-08-25T00:00:00"
SWARM_END   = "2023-08-31T23:59:00"

# outputs
OUTFIG_PDF = "Fig_DAS_vs_SEIS_tol10s_paper.pdf"
OUTFIG_PNG = "Fig_DAS_vs_SEIS_tol10s_paper.png"

# =============================
# LOAD + NORMALIZE
# =============================
df_das  = pd.read_csv(DAS_CSV)
df_seis = pd.read_csv(SEIS_CSV)

# --- DAS
das_time_col = "UTC" if "UTC" in df_das.columns else ("t_abs_utc" if "t_abs_utc" in df_das.columns else None)
if das_time_col is None:
    raise KeyError(f"DAS CSV missing time column (expected 'UTC' or 't_abs_utc'). Columns: {list(df_das.columns)}")
das_cc_col = "CC" if "CC" in df_das.columns else ("cc" if "cc" in df_das.columns else None)

df_das["time"] = pd.to_datetime(df_das[das_time_col], errors="coerce", utc=True).dt.tz_convert(None)
df_das["cc"]   = pd.to_numeric(df_das[das_cc_col], errors="coerce") if das_cc_col else np.nan
df_das = df_das.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)

# --- SEIS
seis_time_col = (
    "event_time_utc" if "event_time_utc" in df_seis.columns else
    ("UTC" if "UTC" in df_seis.columns else
     ("t_abs_utc" if "t_abs_utc" in df_seis.columns else None))
)
if seis_time_col is None:
    raise KeyError(f"SEIS CSV missing time column. Columns: {list(df_seis.columns)}")
seis_cc_col = "cc" if "cc" in df_seis.columns else ("CC" if "CC" in df_seis.columns else None)

df_seis["time"] = pd.to_datetime(df_seis[seis_time_col], errors="coerce", utc=True).dt.tz_convert(None)
df_seis["cc"]   = pd.to_numeric(df_seis[seis_cc_col], errors="coerce") if seis_cc_col else np.nan
df_seis = df_seis.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)

print("DAS detections:", len(df_das))
print("SEIS detections:", len(df_seis))

# =============================
# 1-to-1 greedy nearest matching (±TOL_SEC)
# =============================
tol_ns = np.timedelta64(int(TOL_SEC * 1e9), "ns")

das_times  = df_das["time"].to_numpy(dtype="datetime64[ns]")
seis_times = df_seis["time"].to_numpy(dtype="datetime64[ns]")

used_seis = np.zeros(len(df_seis), dtype=bool)
matched_pairs = []   # (i_das, j_seis, dt_sec)
das_only_idx = []

for i, t in enumerate(das_times):
    j = np.searchsorted(seis_times, t)
    cand = []
    if 0 <= j < len(seis_times):   cand.append(j)
    if 0 <= j-1 < len(seis_times): cand.append(j-1)

    best_j = None
    best_dt = None
    for jj in cand:
        if used_seis[jj]:
            continue
        dt = abs(seis_times[jj] - t)
        if best_dt is None or dt < best_dt:
            best_dt = dt
            best_j = jj

    if best_j is not None and best_dt <= tol_ns:
        used_seis[best_j] = True
        dt_sec = float(best_dt / np.timedelta64(1, "s"))
        matched_pairs.append((i, best_j, dt_sec))
    else:
        das_only_idx.append(i)

seis_only_idx = np.where(~used_seis)[0].tolist()

n_das, n_seis = len(df_das), len(df_seis)
n_m   = len(matched_pairs)
n_do  = len(das_only_idx)
n_so  = len(seis_only_idx)

print(f"Matched (±{TOL_SEC}s): {n_m}")
print(f"DAS only: {n_do}")
print(f"SEIS only: {n_so}")

# =============================
# PAPER STYLE (custom colors + swarm "shadow")
# =============================
# A nicer palette (not matplotlib default blue/orange)
COL_MATCH = "#2A9D8F"   # teal
COL_DAS   = "#6D28D9"   # deep violet
COL_SEIS  = "#111827"   # charcoal
COL_SWARM = "#9CA3AF"   # soft gray

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 350,
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "axes.labelsize": 10.5,
    "axes.titlesize": 12.5,
    "legend.fontsize": 9.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,   # we'll control grids per-axis
})

def format_time_axis(ax):
    loc = mdates.AutoDateLocator(minticks=6, maxticks=10)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    ax.tick_params(axis="x", labelrotation=0)

def add_swarm_shadow(ax, label=True):
    s0 = pd.to_datetime(SWARM_START)
    s1 = pd.to_datetime(SWARM_END)

    # main translucent band
    ax.axvspan(s0, s1, color=COL_SWARM, alpha=0.18, zorder=0)

    # subtle "soft edge" effect: two extra faint bands
    pad = pd.Timedelta(hours=6)
    ax.axvspan(s0-pad, s0, color=COL_SWARM, alpha=0.07, zorder=0)
    ax.axvspan(s1, s1+pad, color=COL_SWARM, alpha=0.07, zorder=0)

    # label
    #if label:
    #    y0, y1 = ax.get_ylim()
    #    ax.text(
    #        s0 + (s1-s0)/2, y1, "Swarm window",
    #        ha="center", va="bottom",
    #        fontsize=9.5, color="#374151"
    #    )

def light_grid(ax):
    ax.grid(True, axis="x", alpha=0.12, linewidth=0.7)
    ax.grid(True, axis="y", alpha=0.10, linewidth=0.6)

# Prepare time arrays
t_matched   = df_das.loc[[i for (i,_,_) in matched_pairs], "time"].to_numpy()
t_das_only  = df_das.loc[das_only_idx, "time"].to_numpy()
t_seis_only = df_seis.loc[seis_only_idx, "time"].to_numpy()

t_das_all  = df_das["time"].to_numpy()
t_seis_all = df_seis["time"].to_numpy()
t_match_sorted = np.sort(t_matched)

# =============================
# FIGURE (2 panels, cleaner + swarm shadow + better styling)
# =============================
fig = plt.figure(figsize=(10.8, 6.4), constrained_layout=True)
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1.0, 1.2])

# ---- Panel A: event raster
ax0 = fig.add_subplot(gs[0, 0])

# add swarm shading (before plotting)
ax0.set_ylim(-0.6, 2.6)
add_swarm_shadow(ax0, label=True)

y_matched, y_das_only, y_seis_only = 0.0, 1.0, 2.0
half = 0.33

# thicker ticks for readability
ax0.vlines(t_matched,   y_matched-half,   y_matched+half,   lw=1.7, color=COL_MATCH, alpha=0.95, zorder=3)
ax0.vlines(t_das_only,  y_das_only-half,  y_das_only+half,  lw=1.7, color=COL_DAS,   alpha=0.92, zorder=3)
ax0.vlines(t_seis_only, y_seis_only-half, y_seis_only+half, lw=1.7, color=COL_SEIS,  alpha=0.85, zorder=3)

ax0.set_yticks([y_matched, y_das_only, y_seis_only])
ax0.set_yticklabels(["Matched", "DAS only", "Stations only"])
#ax0.set_title(f"Detection overlap between DAS and seismic stations (±{TOL_SEC}s)")
ax0.set_xlabel("UTC time")
format_time_axis(ax0)
light_grid(ax0)

# clean legend
handles = [
    plt.Line2D([0],[0], color=COL_MATCH, lw=3, label=f"Matched (n={n_m})"),
    plt.Line2D([0],[0], color=COL_DAS,   lw=3, label=f"DAS only (n={n_do})"),
    plt.Line2D([0],[0], color=COL_SEIS,  lw=3, label=f"Stations only (n={n_so})"),
]
ax0.legend(handles=handles, frameon=True, fancybox=True, framealpha=0.92,
           edgecolor="#E5E7EB", loc="lower right", ncol=3, handlelength=2.2, columnspacing=1.6)

# subtle stats strip (no big box)
ax0.text(
    0.01, 1.02,
    f"DAS={n_das}   Stations={n_seis}   Matched={n_m}   DAS-only={n_do}   Stations-only={n_so}",
    transform=ax0.transAxes, ha="left", va="bottom",
    fontsize=10, color="#374151"
)

ax0.text(-0.02, 1.02, "a", transform=ax0.transAxes, fontweight="bold", va="bottom")

# ---- Panel B: cumulative counts
ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)

# Set ylim first then shade (so label positions behave)
ax1.set_ylim(0, max(len(t_das_all), len(t_seis_all)) * 1.05)
add_swarm_shadow(ax1, label=False)

# step curves
ax1.step(t_seis_all, np.arange(1, len(t_seis_all)+1), where="post", lw=2.4, color=COL_SEIS, label="Stations (cumulative)", alpha=0.95)
ax1.step(t_das_all,  np.arange(1, len(t_das_all)+1),  where="post", lw=2.4, color=COL_DAS,  label="DAS (cumulative)", alpha=0.95)
ax1.step(t_match_sorted, np.arange(1, len(t_match_sorted)+1), where="post", lw=2.6, color=COL_MATCH, label="Matched (cumulative)", alpha=0.98)

#ax1.set_title("Cumulative detections through time")
ax1.set_xlabel("UTC time")
ax1.set_ylabel("Cumulative count")
format_time_axis(ax1)
light_grid(ax1)
ax1.legend(frameon=True, fancybox=True, framealpha=0.92, edgecolor="#E5E7EB", loc="upper left")

# annotate endpoints (small, clean)
ax1.annotate(f"{len(t_seis_all)}", xy=(t_seis_all[-1], len(t_seis_all)), xytext=(6, 0), textcoords="offset points", va="center", color=COL_SEIS)
ax1.annotate(f"{len(t_das_all)}",  xy=(t_das_all[-1],  len(t_das_all)),  xytext=(6, 0), textcoords="offset points", va="center", color=COL_DAS)
ax1.annotate(f"{len(t_match_sorted)}", xy=(t_match_sorted[-1], len(t_match_sorted)), xytext=(6, 0), textcoords="offset points", va="center", color=COL_MATCH)

ax1.text(-0.02, 1.02, "b", transform=ax1.transAxes, fontweight="bold", va="bottom")

# global polish
for ax in (ax0, ax1):
    ax.spines["left"].set_color("#111827")
    ax.spines["bottom"].set_color("#111827")

fig.savefig(OUTFIG_PDF, bbox_inches="tight")
fig.savefig(OUTFIG_PNG, bbox_inches="tight")
plt.show()

print("Saved:", OUTFIG_PDF)
print("Saved:", OUTFIG_PNG)


## 21. Inspect the station table

Display the loaded table to check its columns and event timestamps.


In [ ]:
df_seis

## 22. Explore an extra DAS correlation cutoff

This cell applies CC > 0.070 to create a separate table. Later cells mix filtered and unfiltered DAS variables; reconcile them before using printed match lists as final results.


In [ ]:
#df_das_filter = df_das[df_das["time"] > "2023-11-01"]
df_das_cc_filter = df_das[df_das["CC"] > 0.070]
#df_das_cc_filter_time = df_das_cc_filter[df_das_cc_filter['time']<"2023-08-31"]
#df_das_cc_filter_time

## 23. Retrieve the comparison catalog

Query the 25-km ComCat catalog for August 23 to November 23 at 21:00 UTC. A live query may differ from the archived manuscript catalog; retain the original snapshot for reproducibility.


In [ ]:
# compare this comcat
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

# -----------------------------
# Your requested ComCat window (UTC)
# -----------------------------
COMCAT_START = "2023-08-23T21:00:00"
COMCAT_END   = "2023-11-23T21:00:00"

# Mt. Rainier coords
RAINIER_LAT = 46.879967
RAINIER_LON = -121.726906


# 25 km -> degrees (approx; good enough for small radii)
RADIUS_KM = 25
KM_PER_DEG = 111.19
RADIUS_DEG = RADIUS_KM / KM_PER_DEG   # ~0.225 deg

client = Client("USGS")

def fetch_comcat(start, end):
    """
    Fetch ComCat events within a circular radius around Rainier.
    Note: maxradius is in *degrees* for ObsPy FDSN Client.get_events().
    """
    return client.get_events(
        starttime=UTCDateTime(start),
        endtime=UTCDateTime(end),
        latitude=RAINIER_LAT,
        longitude=RAINIER_LON,
        maxradius=RADIUS_DEG,
        # optional filters if you ever want them:
        # minmagnitude=0.0,
        # orderby="time-asc",
    )

# --- Try a single request first (usually works once radius is correct)
try:
    cat = fetch_comcat(COMCAT_START, COMCAT_END)
except Exception as e:
    # --- Safety fallback: split into chunks (weekly) to avoid server-side 20k limit
    print("Single request failed; falling back to weekly chunks.\nError:", e)

    t0 = pd.to_datetime(COMCAT_START)
    t1 = pd.to_datetime(COMCAT_END)

    chunk_starts = pd.date_range(t0, t1, freq="7D")
    if chunk_starts[-1] != t1:
        chunk_starts = chunk_starts.append(pd.DatetimeIndex([t1]))

    cats = []
    for a, b in zip(chunk_starts[:-1], chunk_starts[1:]):
        c = fetch_comcat(a.isoformat(), b.isoformat())
        cats.append(c)

    # merge catalogs
    cat = cats[0]
    for c in cats[1:]:
        cat += c

# --- Convert to df_comcat
comcat_times = []
for ev in cat:
    if ev.origins:
        comcat_times.append(ev.origins[0].time.datetime)

df_comcat = (
    pd.DataFrame({"time": pd.to_datetime(comcat_times)})
    .dropna(subset=["time"])
    .sort_values("time")
    .reset_index(drop=True)
)

print("df_comcat events:", len(df_comcat))
print("Radius used (deg):", RADIUS_DEG)

## 24. Plot catalog recovery and cumulative counts

Match the loaded tables to ComCat and visualize overlap. A temporal match is a candidate correspondence, not an independently verified source location.


In [ ]:

import matplotlib.dates as mdates

# =============================
# GREEDY 1-to-1 matcher (same style as your DAS–SEIS)
# =============================
def greedy_match(times_A, times_B, tol_sec):
    """
    Match each A to at most one B (and each B used at most once), within ±tol_sec.
    Returns: pairs (iA, jB, dt_sec), A_only_idx, B_only_idx
    """
    tol_ns = np.timedelta64(int(tol_sec * 1e9), "ns")
    A = np.asarray(times_A, dtype="datetime64[ns]")
    B = np.asarray(times_B, dtype="datetime64[ns]")

    used_B = np.zeros(len(B), dtype=bool)
    pairs = []
    A_only = []

    for i, t in enumerate(A):
        j = np.searchsorted(B, t)

        cand = []
        if 0 <= j < len(B):   cand.append(j)
        if 0 <= j-1 < len(B): cand.append(j-1)

        best_j = None
        best_dt = None
        for jj in cand:
            if used_B[jj]:
                continue
            dt = abs(B[jj] - t)
            if best_dt is None or dt < best_dt:
                best_dt = dt
                best_j = jj

        if best_j is not None and best_dt <= tol_ns:
            used_B[best_j] = True
            dt_sec = float(best_dt / np.timedelta64(1, "s"))
            pairs.append((i, best_j, dt_sec))
        else:
            A_only.append(i)

    B_only = np.where(~used_B)[0].tolist()
    return pairs, A_only, B_only


# =============================
# MATCH: COMCAT vs DAS and COMCAT vs SEIS
# =============================
t_cc   = df_comcat["time"].to_numpy(dtype="datetime64[ns]")
t_das  = df_das["time"].to_numpy(dtype="datetime64[ns]")
t_seis = df_seis["time"].to_numpy(dtype="datetime64[ns]")

pairs_cc_das, cc_only_vs_das, das_only_vs_cc = greedy_match(t_cc, t_das, TOL_SEC)
pairs_cc_seis, cc_only_vs_seis, seis_only_vs_cc = greedy_match(t_cc, t_seis, TOL_SEC)

# Build flags on a working comcat table
comcat = df_comcat.copy()
comcat["has_das_match"]  = False
comcat["has_seis_match"] = False
comcat["dt_das_sec"]  = np.nan
comcat["dt_seis_sec"] = np.nan

for i_cc, j_das, dt in pairs_cc_das:
    comcat.loc[i_cc, "has_das_match"] = True
    comcat.loc[i_cc, "dt_das_sec"] = dt

for i_cc, j_seis, dt in pairs_cc_seis:
    comcat.loc[i_cc, "has_seis_match"] = True
    comcat.loc[i_cc, "dt_seis_sec"] = dt

# Categories for plotting
both_mask = comcat["has_das_match"] & comcat["has_seis_match"]
das_mask  = comcat["has_das_match"] & ~comcat["has_seis_match"]
seis_mask = ~comcat["has_das_match"] & comcat["has_seis_match"]
none_mask = ~comcat["has_das_match"] & ~comcat["has_seis_match"]

t_both = comcat.loc[both_mask, "time"].to_numpy()
t_das  = comcat.loc[das_mask,  "time"].to_numpy()
t_seis = comcat.loc[seis_mask, "time"].to_numpy()
t_none = comcat.loc[none_mask, "time"].to_numpy()

print("ComCat total:", len(comcat))
print("ComCat matched by BOTH:", len(t_both))
print("ComCat matched by DAS only:", len(t_das))
print("ComCat matched by SEIS only:", len(t_seis))
print("ComCat only (no TM match):", len(t_none))


# =============================
# PLOT STYLE (reuse your look)
# =============================
COL_BOTH = "#2A9D8F"   # teal (both)
COL_DAS  = "#6D28D9"   # violet (DAS)
COL_SEIS = "#111827"   # charcoal (SEIS)
COL_NONE = "#9CA3AF"   # soft gray (ComCat only)
COL_SWARM = "#9CA3AF"

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 350,
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "axes.labelsize": 10.5,
    "axes.titlesize": 12.5,
    "legend.fontsize": 9.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})

def format_time_axis(ax):
    loc = mdates.AutoDateLocator(minticks=6, maxticks=10)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    ax.tick_params(axis="x", labelrotation=0)

def add_swarm_shadow(ax):
    s0 = pd.to_datetime(SWARM_START)
    s1 = pd.to_datetime(SWARM_END)
    ax.axvspan(s0, s1, color=COL_SWARM, alpha=0.18, zorder=0)
    pad = pd.Timedelta(hours=6)
    ax.axvspan(s0-pad, s0, color=COL_SWARM, alpha=0.07, zorder=0)
    ax.axvspan(s1, s1+pad, color=COL_SWARM, alpha=0.07, zorder=0)

def light_grid(ax):
    ax.grid(True, axis="x", alpha=0.12, linewidth=0.7)
    ax.grid(True, axis="y", alpha=0.10, linewidth=0.6)


# =============================
# FIGURE: 2 panels (raster + cumulative)
# =============================
fig = plt.figure(figsize=(10.8, 6.4), constrained_layout=True)
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1.0, 1.2])

# ---- Panel A: ComCat event raster by category
ax0 = fig.add_subplot(gs[0, 0])
ax0.set_ylim(-0.6, 3.6)
add_swarm_shadow(ax0)

y_both, y_das, y_seis, y_none = 0.0, 1.0, 2.0, 3.0
half = 0.33

ax0.vlines(t_both, y_both-half, y_both+half, lw=1.7, color=COL_BOTH, alpha=0.95, zorder=3)
ax0.vlines(t_das,  y_das-half,  y_das+half,  lw=1.7, color=COL_DAS,  alpha=0.92, zorder=3)
ax0.vlines(t_seis, y_seis-half, y_seis+half, lw=1.7, color=COL_SEIS, alpha=0.85, zorder=3)
ax0.vlines(t_none, y_none-half, y_none+half, lw=1.7, color=COL_NONE, alpha=0.95, zorder=3)

ax0.set_yticks([y_both, y_das, y_seis, y_none])
ax0.set_yticklabels([
    "ComCat matched by BOTH (DAS+SEIS)",
    "ComCat matched by DAS only",
    "ComCat matched by SEIS only",
    "ComCat only (no match)"
])

ax0.set_xlabel("UTC time")
format_time_axis(ax0)
light_grid(ax0)

handles = [
    plt.Line2D([0],[0], color=COL_BOTH, lw=3, label=f"BOTH (n={len(t_both)})"),
    plt.Line2D([0],[0], color=COL_DAS,  lw=3, label=f"DAS only (n={len(t_das)})"),
    plt.Line2D([0],[0], color=COL_SEIS, lw=3, label=f"SEIS only (n={len(t_seis)})"),
    plt.Line2D([0],[0], color=COL_NONE, lw=3, label=f"ComCat only (n={len(t_none)})"),
]
ax0.legend(handles=handles, frameon=True, fancybox=True, framealpha=0.92,
           edgecolor="#E5E7EB", loc="lower right", ncol=2,
           handlelength=2.2, columnspacing=1.6)

ax0.text(
    0.01, 1.02,
    f"ComCat={len(comcat)}   BOTH={len(t_both)}   DAS-only={len(t_das)}   SEIS-only={len(t_seis)}   ComCat-only={len(t_none)}   tol=±{TOL_SEC}s",
    transform=ax0.transAxes, ha="left", va="bottom",
    fontsize=10, color="#374151"
)
ax0.text(-0.02, 1.02, "a", transform=ax0.transAxes, fontweight="bold", va="bottom")

# ---- Panel B: cumulative counts
ax1 = fig.add_subplot(gs[1, 0], sharex=ax0)
add_swarm_shadow(ax1)

# sorted times for steps
t_cc_all = np.sort(comcat["time"].to_numpy())
t_both_s = np.sort(t_both)
t_das_s  = np.sort(t_das)
t_seis_s = np.sort(t_seis)
t_none_s = np.sort(t_none)

ax1.step(t_cc_all,  np.arange(1, len(t_cc_all)+1),  where="post", lw=2.4, color=COL_NONE, label="ComCat (total)", alpha=0.95)
ax1.step(t_none_s,  np.arange(1, len(t_none_s)+1),  where="post", lw=2.4, color=COL_NONE, alpha=0.55, label="ComCat only")
ax1.step(t_seis_s,  np.arange(1, len(t_seis_s)+1),  where="post", lw=2.4, color=COL_SEIS, label="Matched by SEIS (only)", alpha=0.95)
ax1.step(t_das_s,   np.arange(1, len(t_das_s)+1),   where="post", lw=2.4, color=COL_DAS,  label="Matched by DAS (only)", alpha=0.95)
ax1.step(t_both_s,  np.arange(1, len(t_both_s)+1),  where="post", lw=2.6, color=COL_BOTH, label="Matched by BOTH", alpha=0.98)

ax1.set_xlabel("UTC time")
ax1.set_ylabel("Cumulative count")
format_time_axis(ax1)
light_grid(ax1)
ax1.legend(frameon=True, fancybox=True, framealpha=0.92, edgecolor="#E5E7EB", loc="upper left")
ax1.text(-0.02, 1.02, "b", transform=ax1.transAxes, fontweight="bold", va="bottom")

for ax in (ax0, ax1):
    ax.spines["left"].set_color("#111827")
    ax.spines["bottom"].set_color("#111827")

OUTFIG_PDF_COMCAT = "Fig_TM_vs_ComCat_tol{:d}s.pdf".format(TOL_SEC)
OUTFIG_PNG_COMCAT = "Fig_TM_vs_ComCat_tol{:d}s.png".format(TOL_SEC)

fig.savefig(OUTFIG_PDF_COMCAT, bbox_inches="tight")
fig.savefig(OUTFIG_PNG_COMCAT, bbox_inches="tight")
plt.show()


## 25. Inspect event-level matches and unmatched rows

Print the detailed comparisons. Review the mixed df_das and df_das_cc_filter indexing before relying on these exploratory tables. The later regional screening is not implemented here.


In [ ]:
# ============================================
# FULL CELL: compare DAS + SEIS catalogs vs ComCat
# Prints ALL matches + non-matches for BOTH datasets
# (uses your same 1-to-1 greedy nearest matching within ±TOL_SEC)
# ============================================

import numpy as np
import pandas as pd

# -----------------------------
# Sanity checks
# -----------------------------
required = ["df_das_cc_filter", "df_seis", "df_comcat", "TOL_SEC"]
missing = [x for x in required if x not in globals()]
if missing:
    raise NameError(f"Missing variables in notebook: {missing}. "
                    f"Run the load/normalize cells and ComCat fetch cell first.")

for name, df in [("df_das", df_das), ("df_seis", df_seis), ("df_comcat", df_comcat)]:
    if "time" not in df.columns:
        raise KeyError(f"{name} is missing a 'time' column. Columns: {list(df.columns)}")
    if not np.issubdtype(df["time"].dtype, np.datetime64):
        # try to coerce
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
        if df["time"].isna().any():
            raise ValueError(f"{name} has non-parsable times (NaT). Fix time parsing first.")
    df.sort_values("time", inplace=True)
    df.reset_index(drop=True, inplace=True)


# -----------------------------
# Greedy 1-to-1 matcher (A -> B) within tol_sec
# Returns pairs (iA, jB, dt_sec), A_only_idx, B_only_idx
# -----------------------------
def greedy_match(times_A, times_B, tol_sec):
    tol_ns = np.timedelta64(int(tol_sec * 1e9), "ns")
    A = np.asarray(times_A, dtype="datetime64[ns]")
    B = np.asarray(times_B, dtype="datetime64[ns]")

    used_B = np.zeros(len(B), dtype=bool)
    pairs = []
    A_only = []

    for i, t in enumerate(A):
        j = np.searchsorted(B, t)

        cand = []
        if 0 <= j < len(B):   cand.append(j)
        if 0 <= j-1 < len(B): cand.append(j-1)

        best_j = None
        best_dt = None
        for jj in cand:
            if used_B[jj]:
                continue
            dt = abs(B[jj] - t)
            if best_dt is None or dt < best_dt:
                best_dt = dt
                best_j = jj

        if best_j is not None and best_dt <= tol_ns:
            used_B[best_j] = True
            dt_sec = float(best_dt / np.timedelta64(1, "s"))
            pairs.append((i, best_j, dt_sec))
        else:
            A_only.append(i)

    B_only = np.where(~used_B)[0].tolist()
    return pairs, A_only, B_only


# ==========================================================
# MATCH: DAS -> COMCAT   (each DAS uses at most 1 ComCat event)
# ==========================================================
pairs_das_cc, das_not_matched_idx, cc_only_vs_das_idx = greedy_match(
    df_das_cc_filter["time"].to_numpy(dtype="datetime64[ns]"),
    df_comcat["time"].to_numpy(dtype="datetime64[ns]"),
    TOL_SEC
)

# Build DAS matched table (with linked ComCat time + dt)
das_matched = df_das.iloc[[i for (i, _, _) in pairs_das_cc]].copy()
das_matched["i_comcat"] = [j for (_, j, _) in pairs_das_cc]
das_matched["comcat_time"] = df_comcat.iloc[das_matched["i_comcat"].to_numpy()]["time"].to_numpy()
das_matched["dt_sec"] = [dt for (_, _, dt) in pairs_das_cc]
das_matched = das_matched.sort_values("time").reset_index(drop=True)

# DAS not matched
das_not_matched = df_das.iloc[das_not_matched_idx].copy().sort_values("time").reset_index(drop=True)

# ComCat events not used by DAS
comcat_only_vs_das = df_comcat.iloc[cc_only_vs_das_idx].copy().sort_values("time").reset_index(drop=True)


# ==========================================================
# MATCH: SEIS -> COMCAT  (each SEIS uses at most 1 ComCat event)
# ==========================================================
pairs_seis_cc, seis_not_matched_idx, cc_only_vs_seis_idx = greedy_match(
    df_seis["time"].to_numpy(dtype="datetime64[ns]"),
    df_comcat["time"].to_numpy(dtype="datetime64[ns]"),
    TOL_SEC
)

# Build SEIS matched table (with linked ComCat time + dt)
seis_matched = df_seis.iloc[[i for (i, _, _) in pairs_seis_cc]].copy()
seis_matched["i_comcat"] = [j for (_, j, _) in pairs_seis_cc]
seis_matched["comcat_time"] = df_comcat.iloc[seis_matched["i_comcat"].to_numpy()]["time"].to_numpy()
seis_matched["dt_sec"] = [dt for (_, _, dt) in pairs_seis_cc]
seis_matched = seis_matched.sort_values("time").reset_index(drop=True)

# SEIS not matched
seis_not_matched = df_seis.iloc[seis_not_matched_idx].copy().sort_values("time").reset_index(drop=True)

# ComCat events not used by SEIS
comcat_only_vs_seis = df_comcat.iloc[cc_only_vs_seis_idx].copy().sort_values("time").reset_index(drop=True)


# ==========================================================
# SUMMARY PRINTS
# ==========================================================
print("====================================")
print("COMCAT COMPARISON (±{} s)".format(TOL_SEC))
print("====================================\n")

print("DAS detections:", len(df_das_cc_filter))
print("  matched with ComCat:", len(das_matched))
print("  NOT matched with ComCat:", len(das_not_matched))
print()

print("SEIS detections:", len(df_seis))
print("  matched with ComCat:", len(seis_matched))
print("  NOT matched with ComCat:", len(seis_not_matched))
print()

print("ComCat events:", len(df_comcat))
print("  ComCat only vs DAS (no DAS match used):", len(comcat_only_vs_das))
print("  ComCat only vs SEIS (no SEIS match used):", len(comcat_only_vs_seis))
print()

# ==========================================================
# PRINT ALL MATCHES + NON-MATCHES (FULL LISTS)
# ==========================================================
# Choose columns to display robustly (some files might not have cc)
das_cols = ["time"] + (["cc"] if "cc" in df_das_cc_filter.columns else [])
seis_cols = ["time"] + (["cc"] if "cc" in df_seis.columns else [])

print("\n-----------------------------")
print("DAS MATCHED WITH COMCAT (ALL)")
print("-----------------------------")
show_cols = das_cols + ["comcat_time", "dt_sec"]
print(das_matched[show_cols].to_string(index=False))

print("\n---------------------------------")
print("DAS NOT MATCHED WITH COMCAT (ALL)")
print("---------------------------------")
print(das_not_matched[das_cols].to_string(index=False))

print("\n------------------------------")
print("SEIS MATCHED WITH COMCAT (ALL)")
print("------------------------------")
show_cols = seis_cols + ["comcat_time", "dt_sec"]
print(seis_matched[show_cols].to_string(index=False))

print("\n----------------------------------")
print("SEIS NOT MATCHED WITH COMCAT (ALL)")
print("----------------------------------")
print(seis_not_matched[seis_cols].to_string(index=False))



## 26. Historical summary figure, not the revised result

This plot hardcodes 58/62 matches and 27/30 unmatched detections. It predates the wider ComCat screening that changed the station breakdown to 62 local, one regional and 29 additional events. It is retained only for review; do not use it as the final supplementary figure.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Data
# -----------------------------
methods = ["DAS", "Stations"]

matched = np.array([58, 62])       # matched with ComCat
new = np.array([27, 30])           # not in ComCat = new detections
total = matched + new

matched_pct = matched / total * 100
new_pct = new / total * 100

y = np.arange(len(methods))

# -----------------------------
# Colors (muted / paper-style)
# -----------------------------
match_color = "#5F7199"   # muted slate blue
new_color   = "#C7B8A8"   # muted warm beige

# -----------------------------
# Figure
# -----------------------------
fig, ax = plt.subplots(figsize=(6.4, 2.9))

# horizontal stacked bars
ax.barh(y, matched_pct, color=match_color, height=0.44, label="Cataloged in ComCat")
ax.barh(y, new_pct, left=matched_pct, color=new_color, height=0.44, label="New detections")

# -----------------------------
# Labels inside bars + totals
# -----------------------------
for i in range(len(methods)):
    # matched %
    ax.text(
        matched_pct[i] / 2,
        y[i],
        f"{matched_pct[i]:.1f}%",
        ha="center",
        va="center",
        color="white",
        fontsize=9
    )

    # new % + count
    ax.text(
        matched_pct[i] + new_pct[i] / 2,
        y[i],
        f"{new_pct[i]:.1f}%\n({new[i]} new)",
        ha="center",
        va="center",
        color="black",
        fontsize=8.5
    )

    # total detections at right
    ax.text(
        104,
        y[i],
        f"n = {total[i]}",
        ha="left",
        va="center",
        fontsize=9
    )

# -----------------------------
# Axes formatting
# -----------------------------
ax.set_yticks(y)
ax.set_yticklabels(methods, fontsize=10)

ax.set_xlabel("Detection outcome (%)", fontsize=10)
ax.set_xlim(0, 110)

ax.tick_params(axis="x", labelsize=9)
ax.tick_params(axis="y", length=0)

# remove unnecessary spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

# legend above plot
ax.legend(
    frameon=False,
    fontsize=9,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.18),
    ncol=2,
    handlelength=1.8
)

# put DAS on top
ax.invert_yaxis()

plt.tight_layout()

plt.savefig('dasplusstations_2.png',dpi=200)
plt.show()